# Mood’s Median Test

**STA 507 1.0 Nonparametric Statistics** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta507.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Target and construction](#theory-01)
- [Pooled median and ties](#theory-02)
- [Hypotheses and calculation](#theory-03)
- [Conditions and small tables](#theory-04)
- [Relation to other tests](#theory-05)
- [Effect description and conclusion](#theory-06)

## Detailed theory · Mood's median test

<a id="theory-01"></a>

### Target and construction
Mood's median procedure compares independent groups using whether observations fall above or below a common pooled median. It deliberately discards distances and ranks within each side, which can make it less sensitive to shape differences but can reduce power. Observations need an order and a meaningful median definition.

<a id="theory-02"></a>

### Pooled median and ties
Combine observations from all groups and find the pooled median M. Create a group-by-side contingency table. Specify whether observations equal to M are excluded or assigned to a side. The pooled median definition, equal-median ties and group sizes determine the actual table. Different conventions can produce different results.

<a id="theory-03"></a>

### Hypotheses and calculation
The usual chi-square version is an omnibus comparison of the group side probabilities and, under suitable regularity, a test of a common median. For g groups, use expected counts from the row and column margins:
$$X^2=\sum_{j=1}^g\sum_{s=1}^2 (O_{js}-E_{js})^2/E_{js},\qquad df=g-1.$$
Large X² is evidence of unequal side proportions. The usual squared, omnibus version has no signed one-sided direction; a specific directional claim requires a separately justified directed procedure, not simply halving its p-value.

<a id="theory-04"></a>

### Conditions and small tables
Groups and observations must be independent. Adequate expected frequencies are required for the chi-square approximation. With two groups and sparse cells, an exact conditional 2×2 comparison can be used if it matches the intended question; the worked example enumerates the fixed-margin hypergeometric distribution. Report that exact procedure rather than labelling its p-value a chi-square tail.

<a id="theory-05"></a>

### Relation to other tests
Mann–Whitney uses all pooled ranks and is not a distribution-free median-only test under arbitrary unequal shapes. Mood uses only a binary split and is correspondingly coarse. A sign test compares one continuous population median with a fixed target; it is not a general replacement for an independent-group median comparison.

<a id="theory-06"></a>

### Effect description and conclusion
Report group medians and sample sizes, the pooled split point, tie convention, observed and expected counts, statistic, df or exact method, p-value and contextual conclusion. Omnibus significance does not say which pairs differ; planned or adjusted follow-ups are needed for multiple groups. The same handout's two-sample K–S extension is explained in the combined K–S notebook.

## Concepts covered

**Source alignment:** Handout 5: The median test (Mood’s median test).

**Mission:** Match each design and measurement scale to a test, then explain the result in context.

## Worked calculation · Mood median test: pooled median and 2×2 table

### Question and assumptions

Independent groups A=[1,2,3,4,5,6] and B=[7,8,9,10,11,12] have pooled median 6.5. Test whether the groups differ in median, at 5%, using an uncorrected chi-square approximation.

### Procedure

1. Find the pooled median.
2. Classify each value as below or above that median; specify a rule for exact-median ties.
3. Form group-by-side counts and calculate expected counts and chi-square contributions.
4. Check expected frequencies before using an asymptotic cutoff.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Group | Value | Relative to pooled median |
| --- | --- | --- |
| A | 1 | below |
| A | 2 | below |
| A | 3 | below |
| A | 4 | below |
| A | 5 | below |
| A | 6 | below |
| B | 7 | above |
| B | 8 | above |
| B | 9 | above |
| B | 10 | above |
| B | 11 | above |
| B | 12 | above |

| Group | Side | O | E | Contribution |
| --- | --- | --- | --- | --- |
| A | Below | 6 | 3 | 3 |
| A | Above | 0 | 3 | 3 |
| B | Below | 0 | 3 | 3 |
| B | Above | 6 | 3 | 3 |

| Quantity | Value |
| --- | --- |
| Pooled median | 6.5 |
| Chi-square | 12 |
| Smallest E | 3 |
| Approximation suitable? | No: expected counts are only 3 |

| Below-median count in A | Fixed-margin null probability |
| --- | --- |
| 0 | 0.00108225 |
| 1 | 0.038961 |
| 2 | 0.243506 |
| 3 | 0.4329 |
| 4 | 0.243506 |
| 5 | 0.038961 |
| 6 | 0.00108225 |

| Exact conditional 2×2 inference | Value |
| --- | --- |
| Two-sided probability-ordering p | 0.0021645 |
| Decision at .05 | Reject equal side probabilities |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


A=[1,2,3,4,5,6]; B=[7,8,9,10,11,12]; m=median(A+B)
rows=[[g,x,"below" if x<m else "above" if x>m else "tie"] for g,values in [("A",A),("B",B)] for x in values]
show_table(["Group","Value","Relative to pooled median"],rows)
O=[[sum(x<m for x in v),sum(x>m for x in v)] for v in [A,B]]
rt=list(map(sum,O)); ct=[sum(row[j] for row in O) for j in range(2)]; N=sum(rt)
detail=[]
for i in range(2):
    for j in range(2):
        E=rt[i]*ct[j]/N; detail.append(["AB"[i],["Below","Above"][j],O[i][j],E,(O[i][j]-E)**2/E])
show_table(["Group","Side","O","E","Contribution"],detail)
chi=sum(r[-1] for r in detail)
show_table(["Quantity","Value"],[["Pooled median",m],["Chi-square",chi],["Smallest E",min(r[3] for r in detail)],["Approximation suitable?","No: expected counts are only 3"]])
rowA=rt[0]; below=ct[0]; observed=O[0][0]
probabilities=[]
for k in range(max(0,rowA-(N-below)),min(rowA,below)+1):
    probability=comb(below,k)*comb(N-below,rowA-k)/comb(N,rowA)
    probabilities.append([k,probability])
observed_probability=next(p for k,p in probabilities if k==observed)
exact_p=sum(p for k,p in probabilities if p<=observed_probability+1e-12)
show_table(["Below-median count in A","Fixed-margin null probability"],probabilities)
show_table(["Exact conditional 2×2 inference","Value"],[
    ["Two-sided probability-ordering p",exact_p],
    ["Decision at .05","Reject equal side probabilities" if exact_p<=.05 else "Do not reject"]])
assert chi==12
assert abs(sum(p for k,p in probabilities)-1)<1e-12


### Decision and interpretation

Expected counts are only 3, so the asymptotic chi-square cutoff is not used for the decision. The final table instead conditions on margins and sums hypergeometric outcomes whose probabilities are no greater than the observed outcome probability (the two-sided Fisher convention). This is an exact 2×2 comparison after the median categorization, yielding p=2/924≈.002165. The groups differ in above/below-median proportions in this example. Document whether observations equal to the pooled median are excluded, assigned below, or assigned above.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(507)

In [ ]:
a=rng.normal(50,8,24); b=np.r_[rng.normal(50,4,12),rng.normal(50,14,12)]
stat,p,med,table=stats.median_test(a,b, ties='below')
print('grand median:',round(med,2),'above/below table:\n',table)
print('Mood median chi2:',round(stat,3),'p:',round(p,4))
print('group medians:',round(np.median(a),2),round(np.median(b),2))

### Practice · Median workshop
1. What value determines the above/below categories in Mood’s test?
2. Why may Mood’s test be preferable to Mann–Whitney when shapes/spreads differ and the target is median?
3. What are two limitations of the method?

<details><summary>Answer sketches</summary>
1. The pooled grand median. 2. It bases the comparison on sides of a common median rather than all pooled ranks, reducing sensitivity to shape/spread differences. 3. It discards distance/rank information, can have low power, and needs a tie convention and adequate expected counts.
</details>

## Test-selection map

| Question and design | Course method | Key condition |
|---|---|---|
| One-sample success probability | Exact binomial | Independent Bernoulli trials |
| Paired/one-sample location, symmetric differences | Wilcoxon signed-rank | Random sample; symmetric differences |
| Paired/one-sample median, direction only | Sign test | Independent nonzero signs |
| One sample vs specified continuous distribution | One-sample K–S | Fully specified continuous null |
| Counts vs expected categories | Chi-square goodness-of-fit | Independent counts; adequate expected frequencies |
| Two categorical variables / group proportions | Chi-square independence or homogeneity | Independent observations; adequate expected counts |
| Ordered binary sequence | Runs test | Original order retained |
| Two independent ordinal/continuous groups | Mann–Whitney U | Independent samples; similar shapes for median shift |
| Two or more group medians with shape differences | Mood’s median | Independent groups; tie handling; adequate expected counts |

## Final challenge · Choose before computing
For each study, name the response scale, sample relationship, null, and a suitable method: (a) pre/post scores for same patients, symmetric changes; (b) ordered coin tosses; (c) two independent account types and ordinal satisfaction; (d) a sample checked against a specified continuous distribution.

<details><summary>Answer sketches</summary>
(a) Wilcoxon signed-rank (or paired t if its assumptions are defensible); H₀ centered difference 0. (b) Runs test for randomness, preserving order; a binomial test is for the number of heads, not ordering. (c) Mood’s median or Mann–Whitney according to the target and shape assumptions; chi-square on categorized counts may be an alternative. (d) One-sample K–S if null is fully specified and continuous.
</details>